# Train YOLOv11 phát hiện đuối nước trên Kaggle

Dataset: **Swimming and Drowning Detection (Roboflow, 3 lớp)** — `Drowning`, `Person out of water`, `Swimming`.
Code: https://github.com/VuongQuocHung/Real-Time-Drowning-Detection-YOLOv11

**Trước khi chạy:**
1. *Settings → Accelerator*: chọn **GPU T4 x2** hoặc **GPU P100**.
2. *Settings → Internet*: **On** (để `pip install` và `git clone`).
3. *Add Input*: thêm dataset đã upload (thư mục có `data.yaml`, `train/`, `valid/`, `test/`).

Chạy lần lượt từng cell (hoặc *Run All*).

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Cài thư viện

In [ ]:
# lap: cần cho tracker ByteTrack trong main.py
!pip install -q "ultralytics>=8.3.0" lap

## 3. Clone repo

In [ ]:
import os

REPO_URL = "https://github.com/VuongQuocHung/Real-Time-Drowning-Detection-YOLOv11.git"
REPO_DIR = "/kaggle/working/Real-Time-Drowning-Detection-YOLOv11"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q

!ls {REPO_DIR}

## 4. Tìm dataset trong `/kaggle/input`

Tự tìm file `data.yaml` có thư mục `train/images` đi kèm, nên không cần biết tên dataset trên Kaggle.

In [ ]:
from pathlib import Path
import yaml

candidates = [p.parent for p in Path("/kaggle/input").rglob("data.yaml")
              if (p.parent / "train" / "images").is_dir()]
assert candidates, "Không tìm thấy dataset! Kiểm tra lại đã Add Input chưa."
DATASET_DIR = candidates[0]
print("Dataset:", DATASET_DIR)

with open(DATASET_DIR / "data.yaml") as f:
    src_cfg = yaml.safe_load(f)
NAMES = src_cfg["names"]
print("Classes:", NAMES)

# Roboflow đặt tên "valid", Ultralytics hay dùng "val"
VAL_SPLIT = "valid" if (DATASET_DIR / "valid").is_dir() else "val"
for split in ["train", VAL_SPLIT, "test"]:
    n = len(list((DATASET_DIR / split / "images").iterdir()))
    print(f"{split:>6}: {n} ảnh")

## 5. Tạo `data.yaml` mới

`data.yaml` của Roboflow dùng đường dẫn tương đối (`../train/images`). Tạo bản mới với đường dẫn tuyệt đối
trỏ thẳng vào `/kaggle/input` (không cần copy ảnh) và lưu ở `/kaggle/working` vì `/kaggle/input` là read-only.

In [ ]:
WORK_DIR = Path("/kaggle/working/data")
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML = WORK_DIR / "data.yaml"
cfg = {
    "train": str(DATASET_DIR / "train" / "images"),
    "val": str(DATASET_DIR / VAL_SPLIT / "images"),
    "test": str(DATASET_DIR / "test" / "images"),
    "nc": len(NAMES),
    "names": NAMES,
}
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
print(DATA_YAML.read_text())

## 6. Cấu hình train

- `IMGSZ = 416`: Roboflow đã resize mọi ảnh về 416x416, train 640 chỉ chậm hơn mà không thêm thông tin.
- `flipud = 0.0`: tắt lật dọc khi augment (camera hồ bơi không quay ngược); train đã có sẵn ảnh lật dọc từ Roboflow.
- Kaggle giới hạn mỗi phiên ~12 giờ; nếu bị ngắt, đặt `RESUME = True` và chạy lại từ cell này.

In [ ]:
import torch

MODEL = "yolo11n.pt"     # đổi "yolo11s.pt" nếu muốn chính xác hơn (chậm hơn ~2-3 lần)
EPOCHS = 80
IMGSZ = 416
BATCH = 32
PATIENCE = 15
RUN_NAME = "yolo11_swim_drown"
PROJECT = "/kaggle/working/runs"
RESUME = False

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 7. Train

In [ ]:
from ultralytics import YOLO

if RESUME:
    model = YOLO(f"{PROJECT}/{RUN_NAME}/weights/last.pt")
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        optimizer="SGD",
        lr0=0.01,
        patience=PATIENCE,
        device=DEVICE,
        workers=4,          # Kaggle có 4 CPU
        flipud=0.0,
        project=PROJECT,
        name=RUN_NAME,
        exist_ok=True,
        verbose=True,
    )

BEST = Path(PROJECT) / RUN_NAME / "weights" / "best.pt"
print("Best weights:", BEST)

## 8. Đánh giá trên tập test

In [ ]:
best_model = YOLO(str(BEST))
metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=PROJECT,
    name=f"{RUN_NAME}_test",
    exist_ok=True,
)

print("\n===== TEST SET =====")
print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
for i, cls in enumerate(metrics.box.ap_class_index):
    p, r, ap50, _ = metrics.box.class_result(i)
    print(f"{best_model.names[int(cls)]:>20}: precision={p:.3f} recall={r:.3f} mAP50={ap50:.3f}")

## 9. Biểu đồ train và confusion matrix

In [ ]:
from IPython.display import Image as IPImage, display

for f in [Path(PROJECT) / RUN_NAME / "results.png",
          Path(PROJECT) / f"{RUN_NAME}_test" / "confusion_matrix_normalized.png"]:
    if f.exists():
        print(f.name)
        display(IPImage(filename=str(f), width=900))

## 10. Chạy thử `main.py` (tracking + phân tích theo thời gian)

Để thử trên video: upload video vào dataset (hoặc một dataset khác) rồi điền đường dẫn vào `VIDEO_PATH`.
Kết quả lưu ở `output_images/` trong repo. `--no-show` vì Kaggle không có màn hình.

In [ ]:
VIDEO_PATH = None   # ví dụ: "/kaggle/input/drowning-videos/test.mp4"

if VIDEO_PATH:
    !cd {REPO_DIR} && python main.py --model {BEST} --source "{VIDEO_PATH}" --no-show
else:
    # Không có video: chạy thử chế độ ảnh trên vài ảnh test
    for p in sorted((DATASET_DIR / "test" / "images").iterdir())[:3]:
        !cd {REPO_DIR} && python main.py --model {BEST} --source "{p}" --no-show

!ls -la {REPO_DIR}/output_images

In [ ]:
# Xem ảnh kết quả
for f in sorted(Path(REPO_DIR, "output_images").glob("*.jpg"))[:3]:
    display(IPImage(filename=str(f), width=500))

## 11. Đóng gói kết quả để tải về

Tải ở tab *Output* bên phải (hoặc *Save Version* → *Save & Run All* để giữ output lâu dài).
Copy `best.pt` vào thư mục `models/` của repo trên máy để dùng với `main.py`.

In [ ]:
import shutil

shutil.copy(BEST, "/kaggle/working/best.pt")
shutil.make_archive("/kaggle/working/runs", "zip", PROJECT)
!ls -la /kaggle/working